In [2]:
import mlflow
import pandas as pd
import mlflow.sklearn
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd
import re
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import numpy as np

f:\Conda\envs\atlas\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [18]:
df = pd.read_csv('IMDB.csv')
df = df.sample(500)
df.to_csv('data.csv', index=False)
df.head()

,review,sentiment
737,I think this is what this movie wants us to sa...,positive
236,Maybe it's just a personal affection for this ...,positive
549,After seeing The Aristocats: Special Edition i...,positive
881,"or: It's a bird ? It's a plane ? No, look... I...",negative
21,What on earth has become of our dear Ramu? Is ...,negative


In [19]:
# data preprocessing

# Define text preprocessing functions
def lemmatization(text):
    """Lemmatize the text."""
    lemmatizer = WordNetLemmatizer()
    text = text.split()
    text = [lemmatizer.lemmatize(word) for word in text]
    return " ".join(text)

def remove_stop_words(text):
    """Remove stop words from the text."""
    stop_words = set(stopwords.words("english"))
    text = [word for word in str(text).split() if word not in stop_words]
    return " ".join(text)

def removing_numbers(text):
    """Remove numbers from the text."""
    text = ''.join([char for char in text if not char.isdigit()])
    return text

def lower_case(text):
    """Convert text to lower case."""
    text = text.split()
    text = [word.lower() for word in text]
    return " ".join(text)

def removing_punctuations(text):
    """Remove punctuations from the text."""
    text = re.sub('[%s]' % re.escape(string.punctuation), ' ', text)
    text = text.replace('؛', "")
    text = re.sub('\s+', ' ', text).strip()
    return text

def removing_urls(text):
    """Remove URLs from the text."""
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    return url_pattern.sub(r'', text)

def normalize_text(df):
    """Normalize the text data."""
    try:
        df['review'] = df['review'].apply(lower_case)
        df['review'] = df['review'].apply(remove_stop_words)
        df['review'] = df['review'].apply(removing_numbers)
        df['review'] = df['review'].apply(removing_punctuations)
        df['review'] = df['review'].apply(removing_urls)
        df['review'] = df['review'].apply(lemmatization)
        return df
    except Exception as e:
        print(f'Error during text normalization: {e}')
        raise

In [20]:
import nltk

nltk.download("wordnet")
nltk.download("omw-1.4")   # recommended
nltk.download("stopwords") # since you're using stopwords too

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Lenovo\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\Lenovo\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Lenovo\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [21]:
df = normalize_text(df)
df.head()

,review,sentiment
737,think movie want u say end movie damn australi...,positive
236,maybe personal affection screen version mika w...,positive
549,seeing aristocats special edition two pack fox...,positive
881,or bird plane no look disaster or need look sk...,negative
21,earth become dear ramu man made sarkar satya c...,negative


In [22]:
df['sentiment'].value_counts()

sentiment
positive    255
negative    245
Name: count, dtype: int64

In [23]:
x = df['sentiment'].isin(['positive', 'negative'])
df = df[x]

In [24]:
df.head()

,review,sentiment
737,think movie want u say end movie damn australi...,positive
236,maybe personal affection screen version mika w...,positive
549,seeing aristocats special edition two pack fox...,positive
881,or bird plane no look disaster or need look sk...,negative
21,earth become dear ramu man made sarkar satya c...,negative


In [25]:
df['sentiment'] = df['sentiment'].map({'positive':1, 'negative':0})
df.head()

,review,sentiment
737,think movie want u say end movie damn australi...,1
236,maybe personal affection screen version mika w...,1
549,seeing aristocats special edition two pack fox...,1
881,or bird plane no look disaster or need look sk...,0
21,earth become dear ramu man made sarkar satya c...,0


In [26]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [31]:
vectorizer = CountVectorizer(max_features=50)
X = vectorizer.fit_transform(df['review'])
y = df['sentiment']

In [32]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [33]:
import dagshub

mlflow.set_tracking_uri('https://dagshub.com/tushar.dataexpert/End-to-end-mlops-platform.mlflow')
dagshub.init(repo_owner='tushar.dataexpert', repo_name='End-to-end-mlops-platform', mlflow=True)

# mlflow.set_experiment("Logistic Regression Baseline")
mlflow.set_experiment("Logistic Regression Baseline")


2026-05-10 17:26:54,508 - INFO - HTTP Request: GET https://dagshub.com/api/v1/repos/tushar.dataexpert/End-to-end-mlops-platform "HTTP/1.1 200 OK"


Initialized MLflow to track repo "tushar.dataexpert/End-to-end-mlops-platform"

2026-05-10 17:26:54,511 - INFO - Initialized MLflow to track repo "tushar.dataexpert/End-to-end-mlops-platform"


Repository tushar.dataexpert/End-to-end-mlops-platform initialized!

2026-05-10 17:26:54,524 - INFO - Repository tushar.dataexpert/End-to-end-mlops-platform initialized!


<Experiment: artifact_location='mlflow-artifacts:/e03d0b7b00314357a7316908d1363cd3', creation_time=1778413730966, experiment_id='0', last_update_time=1778413730966, lifecycle_stage='active', name='Logistic Regression Baseline', tags={}, trace_location=None, workspace='default'>

In [34]:
import mlflow
import logging
import os
import time
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Configure logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

logging.info("Starting MLflow run...")

with mlflow.start_run():
    start_time = time.time()
    
    try:
        logging.info("Logging preprocessing parameters...")
        mlflow.log_param("vectorizer", "Bag of Words")
        mlflow.log_param("num_features", 50)
        mlflow.log_param("test_size", 0.2)

        logging.info("Initializing Logistic Regression model...")
        model = LogisticRegression(max_iter=1000)  # Increase max_iter to prevent non-convergence issues

        logging.info("Fitting the model...")
        model.fit(X_train, y_train)
        logging.info("Model training complete.")

        logging.info("Logging model parameters...")
        mlflow.log_param("model", "Logistic Regression")

        logging.info("Making predictions...")
        y_pred = model.predict(X_test)

        logging.info("Calculating evaluation metrics...")
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)

        logging.info("Logging evaluation metrics...")
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("f1_score", f1)

        logging.info("Saving and logging the model...")
        mlflow.sklearn.log_model(model, "model")

        # Log execution time
        end_time = time.time()
        logging.info(f"Model training and logging completed in {end_time - start_time:.2f} seconds.")

        # Save and log the notebook
        # notebook_path = "exp1_baseline_model.ipynb"
        # logging.info("Executing Jupyter Notebook. This may take a while...")
        # os.system(f"jupyter nbconvert --to notebook --execute --inplace {notebook_path}")
        # mlflow.log_artifact(notebook_path)

        # logging.info("Notebook execution and logging complete.")

        # Print the results for verification
        logging.info(f"Accuracy: {accuracy}")
        logging.info(f"Precision: {precision}")
        logging.info(f"Recall: {recall}")
        logging.info(f"F1 Score: {f1}")

    except Exception as e:
        logging.error(f"An error occurred: {e}", exc_info=True)


2026-05-10 17:26:55,512 - INFO - Starting MLflow run...
2026-05-10 17:26:56,072 - INFO - Logging preprocessing parameters...
2026-05-10 17:26:57,144 - INFO - Initializing Logistic Regression model...
2026-05-10 17:26:57,144 - INFO - Fitting the model...
2026-05-10 17:26:57,169 - INFO - Model training complete.
2026-05-10 17:26:57,171 - INFO - Logging model parameters...
2026-05-10 17:26:57,516 - INFO - Making predictions...
2026-05-10 17:26:57,518 - INFO - Calculating evaluation metrics...
2026-05-10 17:26:57,537 - INFO - Logging evaluation metrics...
2026-05-10 17:26:58,819 - INFO - Saving and logging the model...
2026/05/10 17:26:58 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/10 17:27:03 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The rec

🏃 View run charming-kite-91 at: https://dagshub.com/tushar.dataexpert/End-to-end-mlops-platform.mlflow/#/experiments/0/runs/db1675d6508d49f6a4b482a9fe9969ab
🧪 View experiment at: https://dagshub.com/tushar.dataexpert/End-to-end-mlops-platform.mlflow/#/experiments/0
